##### Cross-language timing and Laplace consistency | Rectangle

In [ ]:
import numpy as np
from pathlib import Path
from time import perf_counter

# -----------------------------
# Imports (Rectangle)
# -----------------------------
from RectangleGravitationalField.rectangle_potential import potential_batch_rectangle
from RectangleGravitationalField.rectangle_acceleration import acceleration_batch_rectangle
from RectangleGravitationalField.rectangle_tensor import tensor_batch_rectangle


# ============================================================
# CONFIG
# ============================================================

PROJECT_DIR = Path.cwd()   # A_Gravitational_Field_Polygon
RESULTS_DIR = PROJECT_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

# Output files
TXT_OUT = RESULTS_DIR / "Benchmark_Rectangle_Python_vs_MATLAB__PYTHON_RESULTS.txt"

PTS_SPECIAL_CSV = RESULTS_DIR / "benchmark_rectangle_points_special.csv"

PTS50K_CSV = RESULTS_DIR / "benchmark_rectangle_points50000.csv"

# Geometry / physics
L = 1.0
B = 1.0
G = 1.0

D = 1e-7
rho = 1.0
sigma = 2.0 * D * rho

# Tensor params (lamina)
z_r = 0.0
rho_surf = sigma

# Epsilon handling in your rectangle code
eps = 1e-15

# Random benchmark
N_RAND = 50_000
RNG_SEED = 12345

# Domain for random points
XY_MIN, XY_MAX = -2.0, 2.0
Z_MIN, Z_MAX = -2.0, 2.0

# Avoid z=0 for timing + Laplace stability
Z_AVOID = 1e-3


# ============================================================
# POINTS (special cases + one arbitrary off-axis/off-plane)
# ============================================================

points_special = {
    "1) Vertex of rectangle (L,B,0)": ( L,  B, 0.0),
    "2) On edge (x=L, y=0, z=0)":     ( L,  0.0, 0.0),
    "3) Inside on plane (0,0,0)":     ( 0.0, 0.0, 0.0),
    "4) Exterior on plane (2,0,0)":   ( 2.0, 0.0, 0.0),
    "5) Extended edge line (L,2,0)":  ( L,  2.0, 0.0),
    "6) Above (0,0,+1)":              ( 0.0, 0.0, 1.0),
    "7) Below (0,0,-1)":              ( 0.0, 0.0,-1.0),

    # NEW: off-axis, off-plane (exercises Vxy, Vxz, Vyz)
    "8) Off-axis, off-plane (0.37,-0.41,0.83)": (0.37, -0.41, 0.83),
}

names = list(points_special.keys())
pts_special = np.array([points_special[k] for k in names], dtype=float)
N_SPECIAL = pts_special.shape[0]


# ============================================================
# HELPERS
# ============================================================

def fmt(x: float) -> str:
    """Machine-precision-ish formatting; preserves inf/nan."""
    x = float(x)
    if np.isnan(x): return "nan"
    if np.isposinf(x): return "+inf"
    if np.isneginf(x): return "-inf"
    return f"{x:.17e}"

def save_csv_points(path: Path, pts: np.ndarray):
    header = "x,y,z"
    np.savetxt(path, pts, delimiter=",", header=header, comments="", fmt="%.17e")

def make_random_points(N: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    x = rng.uniform(XY_MIN, XY_MAX, size=N)
    y = rng.uniform(XY_MIN, XY_MAX, size=N)
    z = rng.uniform(Z_MIN, Z_MAX, size=N)

    # enforce |z| >= Z_AVOID
    small = np.abs(z) < Z_AVOID
    z[small] = np.where(z[small] >= 0.0, Z_AVOID, -Z_AVOID)

    return np.column_stack([x, y, z]).astype(float)

def eval_rectangle_all(pts: np.ndarray):
    """Return V, (gx,gy,gz), (Vxx,Vyy,Vzz,Vxy,Vxz,Vyz), laplace."""
    V = potential_batch_rectangle(pts, L, B, sigma, G=G, eps=eps)
    gx, gy, gz = acceleration_batch_rectangle(pts, L, B, sigma, G=G, eps=eps)
    Vxx, Vyy, Vzz, Vxy, Vxz, Vyz = tensor_batch_rectangle(
        pts, L, B, z_r, G, rho_surf
    )
    lap = Vxx + Vyy + Vzz
    return V, (gx, gy, gz), (Vxx, Vyy, Vzz, Vxy, Vxz, Vyz), lap


# ============================================================
# RUN: SAVE POINTS FOR MATLAB and Julia
# ============================================================

save_csv_points(PTS_SPECIAL_CSV, pts_special)

pts50k = make_random_points(N_RAND, RNG_SEED)
save_csv_points(PTS50K_CSV, pts50k)


# ============================================================
# RUN: SPECIAL-POINT EVALUATION (FULL OUTPUT)
# ============================================================

Vsp, (gxsp, gysp, gzsp), (Vxxsp, Vyysp, Vzzsp, Vxysp, Vxzsp, Vyzsp), lapsp = eval_rectangle_all(pts_special)


# ============================================================
# RUN: 50k TIMING (SEPARATE V, g, tensor, total)
# ============================================================

t0 = perf_counter()
V50 = potential_batch_rectangle(pts50k, L, B, sigma, G=G, eps=eps)
tV = perf_counter() - t0

t0 = perf_counter()
gx50, gy50, gz50 = acceleration_batch_rectangle(pts50k, L, B, sigma, G=G, eps=eps)
tg = perf_counter() - t0

t0 = perf_counter()
Vxx50, Vyy50, Vzz50, Vxy50, Vxz50, Vyz50 = tensor_batch_rectangle(
    pts50k, L, B, z_r, G, rho_surf
)
tT = perf_counter() - t0

tTotal = tV + tg + tT
lap50 = Vxx50 + Vyy50 + Vzz50

lap_abs_max = np.nanmax(np.abs(lap50))
lap_rms = np.sqrt(np.nanmean(lap50 * lap50))


# ============================================================
# WRITE TXT REPORT (PYTHON RESULTS)
# ============================================================

lines = []
lines.append("Benchmark: Rectangle (Python) vs MATLAB (data exported for MATLAB)\n\n")

lines.append("Directories:\n")
lines.append(f"  Results dir: {RESULTS_DIR}\n")
lines.append(f"  Special-pt CSV: {PTS_SPECIAL_CSV.name}\n")
lines.append(f"  50k CSV:        {PTS50K_CSV.name}\n\n")

lines.append("Parameters:\n")
lines.append(f"  L=B={L}\n")
lines.append(f"  G={G}\n")
lines.append(f"  D={D}\n")
lines.append(f"  rho={rho}\n")
lines.append(f"  sigma=2*D*rho={sigma}\n")
lines.append(f"  z_r={z_r}\n")
lines.append(f"  eps={eps}\n")
lines.append(f"  Random N={N_RAND}, seed={RNG_SEED}\n")
lines.append(f"  Random domain: x,y in [{XY_MIN},{XY_MAX}], z in [{Z_MIN},{Z_MAX}] with |z|>={Z_AVOID}\n\n")

lines.append("="*78 + "\n")
lines.append(f"A) {N_SPECIAL} SPECIAL POINTS (machine precision)\n")
lines.append("="*78 + "\n\n")

for i, name in enumerate(names):
    x, y, z = pts_special[i]
    lines.append(f"{name}\n")
    lines.append(f"  Point (x,y,z) = ({fmt(x)}, {fmt(y)}, {fmt(z)})\n\n")

    lines.append("  Rectangle (Python):\n")
    lines.append(f"    V   = {fmt(Vsp[i])}\n")
    lines.append(f"    gx  = {fmt(gxsp[i])}\n")
    lines.append(f"    gy  = {fmt(gysp[i])}\n")
    lines.append(f"    gz  = {fmt(gzsp[i])}\n\n")

    lines.append("    Tensor:\n")
    lines.append(f"      Vxx = {fmt(Vxxsp[i])}\n")
    lines.append(f"      Vyy = {fmt(Vyysp[i])}\n")
    lines.append(f"      Vzz = {fmt(Vzzsp[i])}\n")
    lines.append(f"      Vxy = {fmt(Vxysp[i])}\n")
    lines.append(f"      Vxz = {fmt(Vxzsp[i])}\n")
    lines.append(f"      Vyz = {fmt(Vyzsp[i])}\n")
    lines.append(f"      Laplace trace (Vxx+Vyy+Vzz) = {fmt(lapsp[i])}\n")

    lines.append("-"*78 + "\n\n")

lines.append("="*78 + "\n")
lines.append("B) 50,000 RANDOM POINTS: TIMING + LAPLACE SUMMARY (Python)\n")
lines.append("="*78 + "\n\n")

lines.append("Timing (seconds):\n")
lines.append(f"  Potential batch:     {tV:.6f} s   ({N_RAND/tV:.1f} pts/s)\n")
lines.append(f"  Acceleration batch:  {tg:.6f} s   ({N_RAND/tg:.1f} pts/s)\n")
lines.append(f"  Tensor batch:        {tT:.6f} s   ({N_RAND/tT:.1f} pts/s)\n")
lines.append(f"  TOTAL:               {tTotal:.6f} s   ({N_RAND/tTotal:.1f} pts/s)\n\n")

lines.append("Laplace trace statistics over random points (should be ~0 outside):\n")
lines.append(f"  max(|Lap|) = {fmt(lap_abs_max)}\n")
lines.append(f"  RMS(Lap)   = {fmt(lap_rms)}\n\n")

TXT_OUT.write_text("".join(lines), encoding="utf-8")

print(f"Saved Python benchmark report: {TXT_OUT}")
print("Exported points for MATLAB:")
print(f"  {PTS_SPECIAL_CSV}")
print(f"  {PTS50K_CSV}")


Rectangle potential: vectorized + parallel (8 CPUs)


/Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Rectangle acceleration: vectorized + parallel (8 CPUs)
Rectangle tensor: vectorized + parallel (8 CPUs)
Saved Python benchmark report: /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/Benchmark_Rectangle_Python_vs_MATLAB__PYTHON_RESULTS.txt
Exported points for MATLAB:
  /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/benchmark_rectangle_points_special.csv
  /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/benchmark_rectangle_points50000.csv


In [2]:
import re
import math
import csv
from pathlib import Path
from itertools import combinations

# ------------------------------------------------------------
# CONFIG (run from A_Gravitational_Field_Polygon/)
# ------------------------------------------------------------
RESULTS_DIR = Path.cwd() / "Results"

REPORTS = {
    "Python": RESULTS_DIR / "Benchmark_Rectangle_Python_vs_MATLAB__PYTHON_RESULTS.txt",
    "MATLAB": RESULTS_DIR / "Benchmark_Rectangle_Python_vs_MATLAB__MATLAB_RESULTS.txt",
    "Julia":  RESULTS_DIR / "Benchmark_Rectangle_Python_vs_MATLAB__JULIA_RESULTS.txt",
}

OUT_TXT = RESULTS_DIR / "Benchmark_Rectangle__PY_MAT_JUL__COMPARISON.txt"
OUT_CSV = RESULTS_DIR / "Benchmark_Rectangle__PY_MAT_JUL__COMPARISON_SPECIALPTS.csv"

FIELDS = [
    "V", "gx", "gy", "gz",
    "Vxx", "Vyy", "Vzz", "Vxy", "Vxz", "Vyz",
    "Laplace",
]

# ------------------------------------------------------------
# UTIL
# ------------------------------------------------------------
def parse_float(s: str) -> float:
    s = s.strip()
    s = s.replace("D", "e").replace("d", "e")
    ls = s.lower()
    if ls in {"+inf", "inf", "infinity", "+infinity"}:
        return float("inf")
    if ls in {"-inf", "-infinity"}:
        return float("-inf")
    if ls in {"nan", "+nan", "-nan"}:
        return float("nan")
    return float(s)

def fmt(x: float) -> str:
    if math.isnan(x): return "nan"
    if math.isinf(x): return "+inf" if x > 0 else "-inf"
    return f"{x:.17e}"

def abs_err(a: float, b: float) -> float:
    return abs(a - b)

def rel_err(ref: float, val: float) -> float:
    # relative error wrt ref; if ref==0 use absolute error
    if math.isnan(ref) or math.isnan(val):
        return float("nan")
    if math.isinf(ref) or math.isinf(val):
        if math.isinf(ref) and math.isinf(val) and (ref > 0) == (val > 0):
            return 0.0
        return float("inf")
    denom = abs(ref)
    if denom == 0.0:
        return abs(ref - val)
    return abs(ref - val) / denom

# ------------------------------------------------------------
# PARSERS
# ------------------------------------------------------------
def parse_points_from_txt(text: str, label: str):
    """
    Returns list of dicts:
      items[i] = {"point": (x,y,z), "V":..., "gx":..., ..., "Laplace":...}
    Parses ALL occurrences of Point (x,y,z) = (...)
    """
    lines = text.splitlines()

    re_point_xyz = re.compile(r"Point\s*\(x,y,z\)\s*=\s*\(([^,]+),\s*([^,]+),\s*([^)]+)\)")
    re_point_xyz_alt = re.compile(r"\(x,y,z\)\s*=\s*\(([^,]+),\s*([^,]+),\s*([^)]+)\)")

    re_val  = re.compile(r"^\s*(V|gx|gy|gz)\s*=\s*(.+)\s*$")
    re_tens = re.compile(r"^\s*(Vxx|Vyy|Vzz|Vxy|Vxz|Vyz)\s*=\s*(.+)\s*$")
    re_lap  = re.compile(r"Laplace.*=\s*(.+)\s*$")

    items = []
    cur = None

    for ln in lines:
        m = re_point_xyz.search(ln) or re_point_xyz_alt.search(ln)
        if m:
            if cur is not None:
                items.append(cur)
            x = parse_float(m.group(1))
            y = parse_float(m.group(2))
            z = parse_float(m.group(3))
            cur = {"point": (x, y, z)}
            continue

        if cur is None:
            continue

        m = re_val.match(ln)
        if m:
            cur[m.group(1)] = parse_float(m.group(2))
            continue

        m = re_tens.match(ln)
        if m:
            cur[m.group(1)] = parse_float(m.group(2))
            continue

        m = re_lap.search(ln)
        if m:
            cur["Laplace"] = parse_float(m.group(1))
            continue

    if cur is not None:
        items.append(cur)

    if len(items) < 1:
        raise RuntimeError(f"{label}: parsed 0 points. Check file format.")
    return items

def parse_timing_and_laplace_stats(text: str, label: str):
    re_time = re.compile(
        r"^\s*(Potential batch|Acceleration batch|Tensor batch|TOTAL)\s*:\s*([0-9.]+)\s*s\s*\(([^)]+)\s*pts/s\)",
        re.IGNORECASE
    )
    re_lap_max = re.compile(r"max\(\|Lap\|\)\s*=\s*([^\s]+)")
    re_lap_rms = re.compile(r"RMS\(Lap\)\s*=\s*([^\s]+)")

    out = {}
    for ln in text.splitlines():
        m = re_time.match(ln)
        if m:
            kind = m.group(1).lower()
            t = float(m.group(2))
            pps = float(m.group(3).strip())
            if "potential" in kind:
                out["tV"] = t; out["ppsV"] = pps
            elif "acceleration" in kind:
                out["tg"] = t; out["ppsg"] = pps
            elif "tensor" in kind:
                out["tT"] = t; out["ppsT"] = pps
            elif "total" in kind:
                out["tTotal"] = t; out["ppsTotal"] = pps

        m = re_lap_max.search(ln)
        if m:
            out["lap_max"] = parse_float(m.group(1))
        m = re_lap_rms.search(ln)
        if m:
            out["lap_rms"] = parse_float(m.group(1))

    needed = ["tV", "tg", "tT", "tTotal", "lap_max", "lap_rms"]
    missing = [k for k in needed if k not in out]
    if missing:
        raise RuntimeError(f"{label}: missing fields in timing/laplace section: {missing}")
    return out

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    # ---- read all reports ----
    texts = {}
    pts = {}
    runs = {}

    for name, path in REPORTS.items():
        if not path.is_file():
            raise FileNotFoundError(f"Missing {name} results file: {path}")
        txt = path.read_text(encoding="utf-8", errors="replace")
        texts[name] = txt
        pts[name] = parse_points_from_txt(txt, f"{name} TXT")
        runs[name] = parse_timing_and_laplace_stats(txt, f"{name} TXT")

    # ---- consistent N ----
    Ns = {name: len(lst) for name, lst in pts.items()}
    N = min(Ns.values())
    if len(set(Ns.values())) != 1:
        print("WARNING: Different #special points detected:")
        for k, v in Ns.items():
            print(f"  {k}: {v}")
        print(f"Comparing first N={N} points.\n")

    # ---- build per-language values table ----
    # rows_values: one row per point-field with values from each lang
    rows_values = []
    for i in range(N):
        # coordinates taken from Python if available else first report
        base_lang = "Python" if "Python" in pts else list(pts.keys())[0]
        x, y, z = pts[base_lang][i]["point"]

        for field in FIELDS:
            row = {
                "point_index": i + 1,
                "x": x, "y": y, "z": z,
                "field": field,
            }
            for lang in REPORTS.keys():
                row[lang] = pts[lang][i].get(field, float("nan"))
            rows_values.append(row)

    # ---- pairwise comparisons (abs & rel wrt first in pair) ----
    pairs = list(combinations(REPORTS.keys(), 2))  # e.g. (Python,MATLAB), ...
    pair_stats = { (a,b): {"max_abs": {f: 0.0 for f in FIELDS},
                           "max_rel": {f: 0.0 for f in FIELDS}} for (a,b) in pairs }

    for row in rows_values:
        for (a, b) in pairs:
            ref = row[a]
            val = row[b]
            ae = abs_err(ref, val)
            re_ = rel_err(ref, val)

            # store per-row errors (useful for CSV)
            row[f"abs_err_{a}_vs_{b}"] = ae
            row[f"rel_err_{a}_vs_{b}"] = re_

            if not math.isnan(ae) and not math.isinf(ae):
                pair_stats[(a,b)]["max_abs"][row["field"]] = max(pair_stats[(a,b)]["max_abs"][row["field"]], ae)
            if not math.isnan(re_) and not math.isinf(re_):
                pair_stats[(a,b)]["max_rel"][row["field"]] = max(pair_stats[(a,b)]["max_rel"][row["field"]], re_)

    # ---- timing comparisons ----
    # speedup(a over b) = ta/tb, like you used before
    timing_pairs = {}
    for (a,b) in pairs:
        timing_pairs[(a,b)] = {
            "Potential": runs[a]["tV"] / runs[b]["tV"],
            "Acceleration": runs[a]["tg"] / runs[b]["tg"],
            "Tensor": runs[a]["tT"] / runs[b]["tT"],
            "TOTAL": runs[a]["tTotal"] / runs[b]["tTotal"],
        }

    # ---- write TXT ----
    out = []
    out.append("Comparison: Rectangle results across Python / MATLAB / Julia\n\n")
    out.append(f"Results dir: {RESULTS_DIR.resolve()}\n")
    for name, path in REPORTS.items():
        out.append(f"{name:6s} file: {path.name}\n")
    out.append("\n")

    out.append("="*78 + "\n")
    out.append(f"A) SPECIAL-POINT NUMERICAL COMPARISON (N={N})\n")
    out.append("="*78 + "\n\n")

    for (a,b) in pairs:
        out.append(f"[Pair] {a} (reference)  vs  {b}\n")
        out.append("  Max abs / rel errors across special points (finite only):\n")
        for f in FIELDS:
            out.append(f"    {f:7s}: max_abs = {fmt(pair_stats[(a,b)]['max_abs'][f])}   "
                       f"max_rel = {fmt(pair_stats[(a,b)]['max_rel'][f])}\n")
        out.append("\n")

    # detailed per point (show values + pairwise errors)
    for i in range(1, N+1):
        # coordinates from first matching row
        r0 = next(r for r in rows_values if r["point_index"] == i and r["field"] == "V")
        out.append(f"Point {i}  (x,y,z)=({fmt(r0['x'])}, {fmt(r0['y'])}, {fmt(r0['z'])})\n")

        for f in FIELDS:
            r = next(r for r in rows_values if r["point_index"] == i and r["field"] == f)
            out.append(f"  {f:7s}  "
                       f"py={fmt(r['Python'])}  ml={fmt(r['MATLAB'])}  jl={fmt(r['Julia'])}\n")
            for (a,b) in pairs:
                out.append(f"           {a} vs {b}: abs={fmt(r[f'abs_err_{a}_vs_{b}'])}  "
                           f"rel={fmt(r[f'rel_err_{a}_vs_{b}'])}\n")
        out.append("-"*78 + "\n\n")

    out.append("="*78 + "\n")
    out.append("B) 50,000-POINT TIMING + LAPLACE STATS COMPARISON\n")
    out.append("="*78 + "\n\n")

    # timings per language
    out.append("Timing (seconds):\n")
    for name in REPORTS.keys():
        out.append(f"  {name:6s}: V={runs[name]['tV']:.6f}  g={runs[name]['tg']:.6f}  "
                   f"T={runs[name]['tT']:.6f}  TOTAL={runs[name]['tTotal']:.6f}\n")
    out.append("\n")

    out.append("Throughput (pts/s):\n")
    for name in REPORTS.keys():
        out.append(f"  {name:6s}: V={runs[name]['ppsV']:.1f}  g={runs[name]['ppsg']:.1f}  "
                   f"T={runs[name]['ppsT']:.1f}  TOTAL={runs[name]['ppsTotal']:.1f}\n")
    out.append("\n")

    out.append("Pairwise speedup (time ratio = time(A)/time(B)):\n")
    for (a,b) in pairs:
        sp = timing_pairs[(a,b)]
        out.append(f"  {a} / {b}:  V={sp['Potential']:.3f}x  g={sp['Acceleration']:.3f}x  "
                   f"T={sp['Tensor']:.3f}x  TOTAL={sp['TOTAL']:.3f}x\n")
    out.append("\n")

    out.append("Laplace trace stats over random points:\n")
    for name in REPORTS.keys():
        out.append(f"  {name:6s}: max(|Lap|)={fmt(runs[name]['lap_max'])}   RMS(Lap)={fmt(runs[name]['lap_rms'])}\n")
    out.append("\n")

    OUT_TXT.write_text("".join(out), encoding="utf-8")
    print(f"Saved comparison TXT: {OUT_TXT}")

    # ---- write CSV ----
    # columns: point, xyz, field, values for each lang, then pairwise errs
    headers = ["point_index", "x", "y", "z", "field"] + list(REPORTS.keys())
    for (a,b) in pairs:
        headers += [f"abs_err_{a}_vs_{b}", f"rel_err_{a}_vs_{b}"]

    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(headers)
        for r in rows_values:
            row_out = [
                r["point_index"],
                f"{r['x']:.17e}", f"{r['y']:.17e}", f"{r['z']:.17e}",
                r["field"],
            ]
            for lang in REPORTS.keys():
                row_out.append(fmt(r[lang]))
            for (a,b) in pairs:
                row_out.append(fmt(r[f"abs_err_{a}_vs_{b}"]))
                row_out.append(fmt(r[f"rel_err_{a}_vs_{b}"]))
            w.writerow(row_out)

    print(f"Saved comparison CSV: {OUT_CSV}")

if __name__ == "__main__":
    main()


Saved comparison TXT: /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/Benchmark_Rectangle__PY_MAT_JUL__COMPARISON.txt
Saved comparison CSV: /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/Benchmark_Rectangle__PY_MAT_JUL__COMPARISON_SPECIALPTS.csv


##### Cross-language timing and Laplace consistency | Triangle

In [1]:
import numpy as np
from pathlib import Path
from time import perf_counter

# -----------------------------
# Imports (Triangle)
# -----------------------------
from TriangleGravitationalField.Triangle_GP import TriangleLaminaGravitation

# ============================================================
# CONFIG
# ============================================================
PROJECT_DIR = Path.cwd()   # A_Gravitational_Field_Polygon
RESULTS_DIR = PROJECT_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

TXT_OUT = RESULTS_DIR / "Benchmark_Triangle_Python_vs_MATLAB__PYTHON_RESULTS.txt"

PTS_SPECIAL_CSV = RESULTS_DIR / "benchmark_triangle_points_special.csv"
PTS50K_CSV      = RESULTS_DIR / "benchmark_triangle_points50000.csv"

# Triangle geometry 
# Here: right triangle in z=0 plane
V0 = np.array([0.0, 0.0, 0.0], dtype=float)
V1 = np.array([1.0, 0.0, 0.0], dtype=float)
V2 = np.array([0.0, 1.0, 0.0], dtype=float)
VERTS = np.vstack([V0, V1, V2])

# Physics
G = 1.0
D = 1.0
rho = 1.0
sigma = 1.0 * D * rho

# eps in your triangle code
eps = 1e-15

tri = TriangleLaminaGravitation(VERTS, G=G, sigma=sigma, eps=eps)

# Random benchmark
N_RAND = 50_000
RNG_SEED = 12345

XY_MIN, XY_MAX = -2.0, 2.0
Z_MIN, Z_MAX = -2.0, 2.0
Z_AVOID = 1e-3

# ============================================================
# SPECIAL POINTS
# (include on-plane, vertex, edge, interior, off-plane, and an arbitrary point)
# ============================================================
points_special = {
    "1) Vertex v0 (0,0,0)": (0.0, 0.0, 0.0),
    "2) Vertex v1 (1,0,0)": (1.0, 0.0, 0.0),
    "3) Vertex v2 (0,1,0)": (0.0, 1.0, 0.0),

    "4) Mid edge v0-v1 (0.5,0,0)": (0.5, 0.0, 0.0),
    "5) Mid edge v0-v2 (0,0.5,0)": (0.0, 0.5, 0.0),
    "6) Interior (0.2,0.2,0)":     (0.2, 0.2, 0.0),

    "7) Above interior (0.2,0.2,+1)": (0.2, 0.2, 1.0),
    "8) Below interior (0.2,0.2,-1)": (0.2, 0.2,-1.0),

    # NEW: off-axis / off-plane point to exercise full tensor (off-diagonals)
    "9) Off-axis, off-plane (0.37,-0.41,0.83)": (0.37, -0.41, 0.83),
}

names = list(points_special.keys())
pts_special = np.array([points_special[k] for k in names], dtype=float)
N_SPECIAL = pts_special.shape[0]

# ============================================================
# HELPERS
# ============================================================
def fmt(x: float) -> str:
    x = float(x)
    if np.isnan(x): return "nan"
    if np.isposinf(x): return "+inf"
    if np.isneginf(x): return "-inf"
    return f"{x:.17e}"

def save_csv_points(path: Path, pts: np.ndarray):
    header = "x,y,z"
    np.savetxt(path, pts, delimiter=",", header=header, comments="", fmt="%.17e")

def make_random_points(N: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    x = rng.uniform(XY_MIN, XY_MAX, size=N)
    y = rng.uniform(XY_MIN, XY_MAX, size=N)
    z = rng.uniform(Z_MIN, Z_MAX, size=N)

    small = np.abs(z) < Z_AVOID
    z[small] = np.where(z[small] >= 0.0, Z_AVOID, -Z_AVOID)

    return np.column_stack([x, y, z]).astype(float)

def eval_triangle_all(pts: np.ndarray):
    V = tri.potential(pts)                 # (N,)
    g = tri.acceleration(pts)              # (N,3)
    T = tri.gravity_tensor(pts)            # (N,3,3) 
    lap = T[:,0,0] + T[:,1,1] + T[:,2,2]   # trace
    return V, g, T, lap

# ============================================================
# RUN: SAVE POINTS FOR MATLAB
# ============================================================
save_csv_points(PTS_SPECIAL_CSV, pts_special)
pts50k = make_random_points(N_RAND, RNG_SEED)
save_csv_points(PTS50K_CSV, pts50k)

# ============================================================
# RUN: SPECIAL POINTS
# ============================================================
Vsp, gsp, Tsp, lapsp = eval_triangle_all(pts_special)

# ============================================================
# RUN: 50k TIMING
# ============================================================
t0 = perf_counter()
V50 = tri.potential(pts50k)
tV = perf_counter() - t0

t0 = perf_counter()
g50 = tri.acceleration(pts50k)
tg = perf_counter() - t0

t0 = perf_counter()
T50 = tri.gravity_tensor(pts50k)
tT = perf_counter() - t0

tTotal = tV + tg + tT
lap50 = T50[:,0,0] + T50[:,1,1] + T50[:,2,2]

lap_abs_max = np.nanmax(np.abs(lap50))
lap_rms = np.sqrt(np.nanmean(lap50 * lap50))

# ============================================================
# WRITE REPORT
# ============================================================
lines = []
lines.append("Benchmark: Triangle (Python) vs MATLAB (data exported for MATLAB)\n\n")

lines.append("Directories:\n")
lines.append(f"  Results dir:      {RESULTS_DIR}\n")
lines.append(f"  Special-pt CSV:   {PTS_SPECIAL_CSV.name}\n")
lines.append(f"  50k CSV:          {PTS50K_CSV.name}\n\n")

lines.append("Triangle vertices:\n")
lines.append(f"  v0 = ({fmt(V0[0])}, {fmt(V0[1])}, {fmt(V0[2])})\n")
lines.append(f"  v1 = ({fmt(V1[0])}, {fmt(V1[1])}, {fmt(V1[2])})\n")
lines.append(f"  v2 = ({fmt(V2[0])}, {fmt(V2[1])}, {fmt(V2[2])})\n\n")

lines.append("Parameters:\n")
lines.append(f"  G={G}\n")
lines.append(f"  D={D}\n")
lines.append(f"  rho={rho}\n")
lines.append(f"  sigma=2*D*rho={sigma}\n")
lines.append(f"  eps={eps}\n")
lines.append(f"  Random N={N_RAND}, seed={RNG_SEED}\n")
lines.append(f"  Random domain: x,y in [{XY_MIN},{XY_MAX}], z in [{Z_MIN},{Z_MAX}] with |z|>={Z_AVOID}\n\n")

lines.append("="*78 + "\n")
lines.append(f"A) {N_SPECIAL} SPECIAL POINTS (machine precision)\n")
lines.append("="*78 + "\n\n")

for i, name in enumerate(names):
    x,y,z = pts_special[i]
    lines.append(f"{name}\n")
    lines.append(f"  Point (x,y,z) = ({fmt(x)}, {fmt(y)}, {fmt(z)})\n\n")

    lines.append("  Triangle (Python):\n")
    lines.append(f"    V   = {fmt(Vsp[i])}\n")
    lines.append(f"    gx  = {fmt(gsp[i,0])}\n")
    lines.append(f"    gy  = {fmt(gsp[i,1])}\n")
    lines.append(f"    gz  = {fmt(gsp[i,2])}\n\n")

    lines.append("    Tensor Γ (NOT symmetrized):\n")
    lines.append(f"      G11 = {fmt(Tsp[i,0,0])}\n")
    lines.append(f"      G22 = {fmt(Tsp[i,1,1])}\n")
    lines.append(f"      G33 = {fmt(Tsp[i,2,2])}\n")
    lines.append(f"      G12 = {fmt(Tsp[i,0,1])}\n")
    lines.append(f"      G13 = {fmt(Tsp[i,0,2])}\n")
    lines.append(f"      G23 = {fmt(Tsp[i,1,2])}\n")
    lines.append(f"      G21 = {fmt(Tsp[i,1,0])}\n")
    lines.append(f"      G31 = {fmt(Tsp[i,2,0])}\n")
    lines.append(f"      G32 = {fmt(Tsp[i,2,1])}\n")
    lines.append(f"      Laplace trace (G11+G22+G33) = {fmt(lapsp[i])}\n")
    lines.append("-"*78 + "\n\n")

lines.append("="*78 + "\n")
lines.append("B) 50,000 RANDOM POINTS: TIMING + LAPLACE SUMMARY (Python)\n")
lines.append("="*78 + "\n\n")

lines.append("Timing (seconds):\n")
lines.append(f"  Potential batch:     {tV:.6f} s   ({N_RAND/tV:.1f} pts/s)\n")
lines.append(f"  Acceleration batch:  {tg:.6f} s   ({N_RAND/tg:.1f} pts/s)\n")
lines.append(f"  Tensor batch:        {tT:.6f} s   ({N_RAND/tT:.1f} pts/s)\n")
lines.append(f"  TOTAL:               {tTotal:.6f} s   ({N_RAND/tTotal:.1f} pts/s)\n\n")

lines.append("Laplace trace statistics over random points (should be ~0 outside):\n")
lines.append(f"  max(|Lap|) = {fmt(lap_abs_max)}\n")
lines.append(f"  RMS(Lap)   = {fmt(lap_rms)}\n\n")

TXT_OUT.write_text("".join(lines), encoding="utf-8")

print(f"Saved Python benchmark report: {TXT_OUT}")
print("Exported points for MATLAB:")
print(f"  {PTS_SPECIAL_CSV}")
print(f"  {PTS50K_CSV}")


Saved Python benchmark report: /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/Benchmark_Triangle_Python_vs_MATLAB__PYTHON_RESULTS.txt
Exported points for MATLAB:
  /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/benchmark_triangle_points_special.csv
  /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/benchmark_triangle_points50000.csv


In [4]:
# compare_benchmark_triangle_py_mat_jul.py
import re
import math
import csv
from pathlib import Path
from itertools import combinations

# ------------------------------------------------------------
# CONFIG (run from A_Gravitational_Field_Polygon/)
# ------------------------------------------------------------
RESULTS_DIR = Path.cwd() / "Results"

REPORTS = {
    "Python": RESULTS_DIR / "Benchmark_Triangle_Python_vs_MATLAB__PYTHON_RESULTS.txt",
    "MATLAB": RESULTS_DIR / "Benchmark_Triangle_Python_vs_MATLAB__MATLAB_RESULTS.txt",
    "Julia":  RESULTS_DIR / "Benchmark_Triangle_Python_vs_MATLAB__JULIA_RESULTS.txt",
}

OUT_TXT = RESULTS_DIR / "Benchmark_Triangle__PY_MAT_JUL__COMPARISON.txt"
OUT_CSV = RESULTS_DIR / "Benchmark_Triangle__PY_MAT_JUL__COMPARISON_SPECIALPTS.csv"

# Triangle tensor naming differs from rectangle:
#   Potential: V
#   Acceleration: gx,gy,gz
#   Tensor: G11..G33 (NOT symmetrized)
#   Laplace: trace = G11+G22+G33
FIELDS = [
    "V", "gx", "gy", "gz",
    "G11", "G22", "G33",
    "G12", "G13", "G23",
    "G21", "G31", "G32",
    "Laplace",
]

# ------------------------------------------------------------
# UTIL
# ------------------------------------------------------------
def parse_float(s: str) -> float:
    s = s.strip()
    s = s.replace("D", "e").replace("d", "e")
    ls = s.lower()
    if ls in {"+inf", "inf", "infinity", "+infinity"}:
        return float("inf")
    if ls in {"-inf", "-infinity"}:
        return float("-inf")
    if ls in {"nan", "+nan", "-nan"}:
        return float("nan")
    return float(s)

def fmt(x: float) -> str:
    if math.isnan(x): return "nan"
    if math.isinf(x): return "+inf" if x > 0 else "-inf"
    return f"{x:.17e}"

def abs_err(a: float, b: float) -> float:
    return abs(a - b)

def rel_err(ref: float, val: float) -> float:
    # relative error wrt ref; if ref==0 use absolute error
    if math.isnan(ref) or math.isnan(val):
        return float("nan")
    if math.isinf(ref) or math.isinf(val):
        if math.isinf(ref) and math.isinf(val) and (ref > 0) == (val > 0):
            return 0.0
        return float("inf")
    denom = abs(ref)
    if denom == 0.0:
        return abs(ref - val)
    return abs(ref - val) / denom

# ------------------------------------------------------------
# PARSERS
# ------------------------------------------------------------
def parse_points_from_txt(text: str, label: str):
    """
    Returns list of dicts:
      items[i] = {"point": (x,y,z), "V":..., "gx":..., ..., "Laplace":...}

    Triangle report formats contain:
      Point (x,y,z) = (...)
      V = ...
      gx = ...
      ...
      G11 = ...
      ...
      Laplace trace ... = ...

    Parses ALL occurrences of Point (x,y,z) = (...)
    """
    lines = text.splitlines()

    re_point_xyz = re.compile(r"Point\s*\(x,y,z\)\s*=\s*\(([^,]+),\s*([^,]+),\s*([^)]+)\)")
    re_point_xyz_alt = re.compile(r"\(x,y,z\)\s*=\s*\(([^,]+),\s*([^,]+),\s*([^)]+)\)")

    re_val  = re.compile(r"^\s*(V|gx|gy|gz)\s*=\s*(.+)\s*$")
    re_tens = re.compile(r"^\s*(G11|G12|G13|G21|G22|G23|G31|G32|G33)\s*=\s*(.+)\s*$")
    re_lap  = re.compile(r"Laplace\s*trace.*=\s*(.+)\s*$", re.IGNORECASE)

    items = []
    cur = None

    for ln in lines:
        m = re_point_xyz.search(ln) or re_point_xyz_alt.search(ln)
        if m:
            if cur is not None:
                items.append(cur)
            x = parse_float(m.group(1))
            y = parse_float(m.group(2))
            z = parse_float(m.group(3))
            cur = {"point": (x, y, z)}
            continue

        if cur is None:
            continue

        m = re_val.match(ln)
        if m:
            cur[m.group(1)] = parse_float(m.group(2))
            continue

        m = re_tens.match(ln)
        if m:
            cur[m.group(1)] = parse_float(m.group(2))
            continue

        m = re_lap.search(ln)
        if m:
            cur["Laplace"] = parse_float(m.group(1))
            continue

    if cur is not None:
        items.append(cur)

    if len(items) < 1:
        raise RuntimeError(f"{label}: parsed 0 points. Check file format.")
    return items

def parse_timing_and_laplace_stats(text: str, label: str):
    """
    Robust timing + Laplace parser.

    Supports BOTH styles:
      Rectangle/Python/Julia style:
        "Potential batch:    0.123456 s   (12345.6 pts/s)"
      MATLAB triangle style:
        "Potential:          0.052348 s   (955142.5 pts/s)"
    """
    # Accept optional "batch" word and allow whitespace alignment.
    re_time = re.compile(
        r"^\s*(Potential(?:\s+batch)?|Acceleration(?:\s+batch)?|Tensor(?:\s+batch)?|TOTAL)\s*:\s*"
        r"([0-9.]+)\s*s\s*\(([^)]+)\s*pts/s\)",
        re.IGNORECASE
    )

    re_lap_max = re.compile(r"max\(\|Lap\|\)\s*=\s*([^\s]+)", re.IGNORECASE)
    re_lap_rms = re.compile(r"RMS\(Lap\)\s*=\s*([^\s]+)", re.IGNORECASE)

    out = {}
    for ln in text.splitlines():
        m = re_time.match(ln)
        if m:
            kind = m.group(1).lower().replace("batch", "").strip()
            t = float(m.group(2))
            pps = float(m.group(3).strip())

            if kind.startswith("potential"):
                out["tV"] = t; out["ppsV"] = pps
            elif kind.startswith("acceleration"):
                out["tg"] = t; out["ppsg"] = pps
            elif kind.startswith("tensor"):
                out["tT"] = t; out["ppsT"] = pps
            elif kind.startswith("total"):
                out["tTotal"] = t; out["ppsTotal"] = pps

        m = re_lap_max.search(ln)
        if m:
            out["lap_max"] = parse_float(m.group(1))

        m = re_lap_rms.search(ln)
        if m:
            out["lap_rms"] = parse_float(m.group(1))

    needed = ["tV", "tg", "tT", "tTotal", "lap_max", "lap_rms"]
    missing = [k for k in needed if k not in out]
    if missing:
        raise RuntimeError(f"{label}: missing fields in timing/laplace section: {missing}")
    return out


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    # ---- read all reports ----
    texts = {}
    pts = {}
    runs = {}

    for name, path in REPORTS.items():
        if not path.is_file():
            raise FileNotFoundError(f"Missing {name} results file: {path}")
        txt = path.read_text(encoding="utf-8", errors="replace")
        texts[name] = txt
        pts[name] = parse_points_from_txt(txt, f"{name} TXT")
        runs[name] = parse_timing_and_laplace_stats(txt, f"{name} TXT")

    # ---- consistent N ----
    Ns = {name: len(lst) for name, lst in pts.items()}
    N = min(Ns.values())
    if len(set(Ns.values())) != 1:
        print("WARNING: Different #special points detected:")
        for k, v in Ns.items():
            print(f"  {k}: {v}")
        print(f"Comparing first N={N} points.\n")

    # ---- build per-point-field values table ----
    rows_values = []
    for i in range(N):
        base_lang = "Python" if "Python" in pts else list(pts.keys())[0]
        x, y, z = pts[base_lang][i]["point"]

        for field in FIELDS:
            row = {
                "point_index": i + 1,
                "x": x, "y": y, "z": z,
                "field": field,
            }
            for lang in REPORTS.keys():
                row[lang] = pts[lang][i].get(field, float("nan"))
            rows_values.append(row)

    # ---- pairwise comparisons ----
    pairs = list(combinations(REPORTS.keys(), 2))  # (Python,MATLAB), ...
    pair_stats = {
        (a, b): {"max_abs": {f: 0.0 for f in FIELDS},
                 "max_rel": {f: 0.0 for f in FIELDS}}
        for (a, b) in pairs
    }

    for row in rows_values:
        for (a, b) in pairs:
            ref = row[a]
            val = row[b]
            ae = abs_err(ref, val)
            re_ = rel_err(ref, val)

            row[f"abs_err_{a}_vs_{b}"] = ae
            row[f"rel_err_{a}_vs_{b}"] = re_

            if not math.isnan(ae) and not math.isinf(ae):
                pair_stats[(a, b)]["max_abs"][row["field"]] = max(pair_stats[(a, b)]["max_abs"][row["field"]], ae)
            if not math.isnan(re_) and not math.isinf(re_):
                pair_stats[(a, b)]["max_rel"][row["field"]] = max(pair_stats[(a, b)]["max_rel"][row["field"]], re_)

    # ---- timing comparisons ----
    timing_pairs = {}
    for (a, b) in pairs:
        timing_pairs[(a, b)] = {
            "Potential": runs[a]["tV"] / runs[b]["tV"],
            "Acceleration": runs[a]["tg"] / runs[b]["tg"],
            "Tensor": runs[a]["tT"] / runs[b]["tT"],
            "TOTAL": runs[a]["tTotal"] / runs[b]["tTotal"],
        }

    # ---- write TXT ----
    out = []
    out.append("Comparison: Triangle results across Python / MATLAB / Julia\n\n")
    out.append(f"Results dir: {RESULTS_DIR.resolve()}\n")
    for name, path in REPORTS.items():
        out.append(f"{name:6s} file: {path.name}\n")
    out.append("\n")

    out.append("=" * 78 + "\n")
    out.append(f"A) SPECIAL-POINT NUMERICAL COMPARISON (N={N})\n")
    out.append("=" * 78 + "\n\n")

    for (a, b) in pairs:
        out.append(f"[Pair] {a} (reference)  vs  {b}\n")
        out.append("  Max abs / rel errors across special points (finite only):\n")
        for f in FIELDS:
            out.append(
                f"    {f:7s}: max_abs = {fmt(pair_stats[(a, b)]['max_abs'][f])}   "
                f"max_rel = {fmt(pair_stats[(a, b)]['max_rel'][f])}\n"
            )
        out.append("\n")

    # detailed per point
    for i in range(1, N + 1):
        r0 = next(r for r in rows_values if r["point_index"] == i and r["field"] == "V")
        out.append(f"Point {i}  (x,y,z)=({fmt(r0['x'])}, {fmt(r0['y'])}, {fmt(r0['z'])})\n")

        for f in FIELDS:
            r = next(r for r in rows_values if r["point_index"] == i and r["field"] == f)
            out.append(
                f"  {f:7s}  py={fmt(r['Python'])}  ml={fmt(r['MATLAB'])}  jl={fmt(r['Julia'])}\n"
            )
            for (a, b) in pairs:
                out.append(
                    f"           {a} vs {b}: abs={fmt(r[f'abs_err_{a}_vs_{b}'])}  "
                    f"rel={fmt(r[f'rel_err_{a}_vs_{b}'])}\n"
                )
        out.append("-" * 78 + "\n\n")

    out.append("=" * 78 + "\n")
    out.append("B) 50,000-POINT TIMING + LAPLACE STATS COMPARISON\n")
    out.append("=" * 78 + "\n\n")

    out.append("Timing (seconds):\n")
    for name in REPORTS.keys():
        out.append(
            f"  {name:6s}: V={runs[name]['tV']:.6f}  g={runs[name]['tg']:.6f}  "
            f"T={runs[name]['tT']:.6f}  TOTAL={runs[name]['tTotal']:.6f}\n"
        )
    out.append("\n")

    out.append("Throughput (pts/s):\n")
    for name in REPORTS.keys():
        out.append(
            f"  {name:6s}: V={runs[name]['ppsV']:.1f}  g={runs[name]['ppsg']:.1f}  "
            f"T={runs[name]['ppsT']:.1f}  TOTAL={runs[name]['ppsTotal']:.1f}\n"
        )
    out.append("\n")

    out.append("Pairwise speedup (time ratio = time(A)/time(B)):\n")
    for (a, b) in pairs:
        sp = timing_pairs[(a, b)]
        out.append(
            f"  {a} / {b}:  V={sp['Potential']:.3f}x  g={sp['Acceleration']:.3f}x  "
            f"T={sp['Tensor']:.3f}x  TOTAL={sp['TOTAL']:.3f}x\n"
        )
    out.append("\n")

    out.append("Laplace trace stats over random points:\n")
    for name in REPORTS.keys():
        out.append(
            f"  {name:6s}: max(|Lap|)={fmt(runs[name]['lap_max'])}   RMS(Lap)={fmt(runs[name]['lap_rms'])}\n"
        )
    out.append("\n")

    OUT_TXT.write_text("".join(out), encoding="utf-8")
    print(f"Saved comparison TXT: {OUT_TXT}")

    # ---- write CSV ----
    headers = ["point_index", "x", "y", "z", "field"] + list(REPORTS.keys())
    for (a, b) in pairs:
        headers += [f"abs_err_{a}_vs_{b}", f"rel_err_{a}_vs_{b}"]

    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(headers)
        for r in rows_values:
            row_out = [
                r["point_index"],
                f"{r['x']:.17e}", f"{r['y']:.17e}", f"{r['z']:.17e}",
                r["field"],
            ]
            for lang in REPORTS.keys():
                row_out.append(fmt(r[lang]))
            for (a, b) in pairs:
                row_out.append(fmt(r[f"abs_err_{a}_vs_{b}"]))
                row_out.append(fmt(r[f"rel_err_{a}_vs_{b}"]))
            w.writerow(row_out)

    print(f"Saved comparison CSV: {OUT_CSV}")

if __name__ == "__main__":
    main()


Saved comparison TXT: /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/Benchmark_Triangle__PY_MAT_JUL__COMPARISON.txt
Saved comparison CSV: /Volumes/Dunendran/Programs/Python/A_Gravitational_Field_Polygon/Results/Benchmark_Triangle__PY_MAT_JUL__COMPARISON_SPECIALPTS.csv
